# Intro til Machine Learning — 4: Ekstra opgaver

Velkommen til slutspillet. Disse opgaver er **større, friere og sværere** end dem i
regressionsforløbet-4, og de kombinerer alt, hvad I har lært. Der er ingen fast rækkefølge — vælg
den, der frister mest. Hver opgave har startkode, så I aldrig starter fra en blank side.

| Opgave | Hvad | Bruger |
|---|---|---|
| E.1 | Pokédex-rapporten | pandas + plots |
| E.2 | Gradient descent i 2D | autograd + GD |
| E.3 | Type-oraklet (18 klasser!) | hele ML-pipelinen |
| E.4 | Den store aktiveringsdyst | MNIST + aktiveringer |
| E.5 | Hyperparameter-jagten | pandas MØDER PyTorch |
| E.6 | Tegn selv et tal | MNIST + kreativitet |
| E.7 | Fejl-detektiven (boss-level ×5) | ALT |
| E.8 | Frit valg fra Kaggle | mod og eventyrlyst |

## Setup (kør altid denne først)

In [ ]:
!pip install -q kagglehub

!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/98-Helpers/helpers.py
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/28-Data/MLData/Pokemon.csv
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/28-Data/MLData/mnist_traen_lille.csv.gz
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/28-Data/MLData/mnist_test_lille.csv.gz

import kagglehub   # bruges kun i E.8 (frit valg fra Kaggle)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

from helpers import show_mnist_images

torch.manual_seed(42)
np.random.seed(42)

# Pokémon (hentet med wget ovenfor)
df = pd.read_csv("Pokemon.csv")

# MNIST (bruges i E.4, E.6 og E.7) — allerede nedskaleret
train_df = pd.read_csv("mnist_traen_lille.csv.gz").sample(n=8000, random_state=42).reset_index(drop=True)
test_df = pd.read_csv("mnist_test_lille.csv.gz").reset_index(drop=True)

X_mnist = torch.tensor(train_df.drop(columns=["label"]).values / 255.0, dtype=torch.float32)
y_mnist = torch.tensor(train_df["label"].values, dtype=torch.long)
X_mnist_test = torch.tensor(test_df.drop(columns=["label"]).values / 255.0, dtype=torch.float32)
y_mnist_test = torch.tensor(test_df["label"].values, dtype=torch.long)

print("klar!", df.shape, X_mnist.shape)

# 1: Store udfordringer

Vælg frit — de kan tages i vilkårlig rækkefølge.


##### Opgave 1.1 — Pokédex-rapporten

I er blevet hyret som dataanalytikere af Professor Oak. Han vil vide: **hvilken generation
er den "bedste"?** Men "bedst" er jeres valg at definere — stærkest i snit? flest
legendariske? bedst balance mellem angreb og forsvar? mest variation?

**Krav til rapporten (byg den i denne notebook):**
1. Mindst **3 forskellige plots** (fx histogram, scatter, søjler)
2. Mindst én **groupby**
3. Mindst én **ny kolonne**, I selv har beregnet
4. En **konklusion i en markdown-celle**: hvilken generation vinder, og hvorfor?

Startkoden giver et første fingerpeg — resten er op til jer.

In [ ]:
# ET eksempel på en rapport (jeres kan se HELT anderledes ud — det er meningen):

# Plot 1: styrke pr. generation
df.groupby("Generation")["Total"].mean().plot(kind="bar")
plt.ylabel("gennemsnitlig Total")
plt.title("Gen 4 er stærkest i snit")
plt.show()

# Plot 2 + ny kolonne: balance mellem offensiv og defensiv
df["Balance"] = (df["Attack"] + df["Sp. Atk"]) / (df["Defense"] + df["Sp. Def"])
df.groupby("Generation")["Balance"].mean().plot(kind="bar", color="orange")
plt.ylabel("offensiv/defensiv-balance")
plt.title("Gen 1 er mest angrebslysten")
plt.show()

# Plot 3: andel legendariske pr. generation
fraction = df.groupby("Generation")["Legendary"].mean()
fraction.plot(kind="bar", color="gold")
plt.ylabel("andel legendariske")
plt.title("Gen 3 og 4 har flest legendariske pr. Pokémon")
plt.show()

# Plot 4: spredning — hvor forskellige er generationens Pokémon?
df.boxplot(column="Total", by="Generation")
plt.suptitle("")
plt.title("Spredningen af Total pr. generation")
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Eksempel-konklusion: Generation 4 vinder på råstyrke (højeste gennemsnitlige Total ≈ 459) og er sammen med gen 3 rigest på legendariske; gen 2 er svagest i snit. Men pointen med opgaven er PROCESSEN: definér "bedst", mål det, og lad plots bære argumentet. Godkend alt, der opfylder de fire krav og har en begrundet konklusion.</span>

##### Opgave 1.2 — Gradient descent i 2D

I regressionsforløbet rullede I ned ad en 1D-bakke. Rigtige tabslandskaber har millioner af
dimensioner — men i 2D kan vi stadig TEGNE det. Funktionen
$f(a, b) = (a-1)^2 + 3(b+2)^2$ er en oval dal med bund i $(1, -2)$.

**Jeres mission:** udfyld gradienten og opdateringsskridtene, og tegn ruten ned i dalen
oven på konturplottet. Prøv derefter forskellige startpunkter og læringsrater. (Bonus:
brug autograd i stedet for hånd-gradienten og tjek, at ruten er den samme.)

In [ ]:
def f(a, b):
    return (a - 1) ** 2 + 3 * (b + 2) ** 2

A, B = np.meshgrid(np.linspace(-4, 6, 100), np.linspace(-6, 2, 100))
plt.contour(A, B, f(A, B), levels=30, cmap="viridis")
plt.colorbar(label="f(a, b)")

for start_a, start_b, color in [(5.0, 1.0, "crimson"), (-3.0, 1.5, "royalblue"), (5.5, -5.5, "green")]:
    a, b = start_a, start_b
    learning_rate = 0.1
    route_a, route_b = [a], [b]
    for i in range(40):
        grad_a = 2 * (a - 1)
        grad_b = 6 * (b + 2)
        a = a - learning_rate * grad_a
        b = b - learning_rate * grad_b
        route_a.append(a)
        route_b.append(b)
    plt.plot(route_a, route_b, "o-", color=color, markersize=4)

plt.scatter([1], [-2], color="gold", s=150, zorder=3, edgecolors="black", label="minimum")
plt.legend()
plt.title("Tre startpunkter — samme dal")
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Bemærk hvordan ruterne bevæger sig hurtigst i b-retningen (koefficienten 3 gør dalen stejlere den vej — gradienten er større) og til sidst kryber langs den flade a-retning. Det er et minieksempel på, hvorfor "forskellige skalaer" (regressionsforløbet!) gør GD kluntet — og hvorfor optimizers som Adam, der skalerer pr. retning, blev opfundet. Autograd-bonus: <code>a = torch.tensor(5.0, requires_grad=True)</code> osv., <code>f(a, b).backward()</code>, og brug <code>a.grad</code>/<code>b.grad</code> — samme rute.</span>

##### Opgave 1.3 — Type-oraklet

Kan man se på en Pokémons stats, hvilken **type** den er? Det er klassifikation med
**18 klasser** — jeres hidtil vildeste. Udfyld hullerne (mønstret er 12.6 + notebook 4).

Bagefter: sammenlign med den dovne baseline (gæt altid på den mest almindelige type —
hvor god er DEN? Husk opgave 2.5!), og diskutér: hvorfor er det her SÅ meget sværere end
at spotte legendariske?

In [ ]:
types = sorted(df["Type 1"].unique())
print(len(types), "typer:", types)
type_til_nr = {t: nr for nr, t in enumerate(types)}

seks_stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
X_np = df[seks_stats].values.astype("float32")
X_np = (X_np - X_np.mean(axis=0)) / X_np.std(axis=0)
y_np = df["Type 1"].map(type_til_nr).values

X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(X_np, y_np, test_size=0.2, random_state=42)
X_train, X_test = torch.tensor(X_train_np), torch.tensor(X_test_np)
y_train = torch.tensor(y_train_np, dtype=torch.long)
y_test = torch.tensor(y_test_np, dtype=torch.long)

class TypeOracle(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(6, 64)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(64, 18)

    def forward(self, x):
        return self.layer2(self.activation(self.layer1(x)))

model = TypeOracle()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
for epoch in range(2000):
    optimizer.zero_grad()
    loss = loss_fn(model(X_train), y_train)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    pred = model(X_test).argmax(dim=1)
print(f"oraklets accuracy:  {(pred == y_test).float().mean().item():.1%}")

mest_almindelige = type_til_nr[df["Type 1"].value_counts().index[0]]   # Water
lazy = (y_test == mest_almindelige).float().mean()
print(f"dovne baseline:     {lazy.item():.1%} (gæt altid Water)")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Oraklet lander typisk omkring 19-25 % — bedre end den dovne baseline (15 %, andelen af Water i testsættet) og LANGT bedre end rent held (1/18 ≈ 5,6 %), men stadig dårligt. Hvorfor så svært? (1) Stats bestemmer ikke typen — en Water-type kan have alle slags stats; informationen er der simpelthen kun delvist. (2) 18 klasser med meget skæv fordeling (Water 112, Flying 4!). (3) Kun ~35 eksempler pr. klasse i snit. Vigtig lærdom: nogle gange ligger begrænsningen i DATAENE, ikke i modellen — der findes intet netværk, der kan trylle information frem, som ikke er der.</span>

##### Opgave 1.4 — Den store aktiveringsdyst

Afgør det én gang for alle (i hvert fald for MNIST): træn det SAMME netværk med **ReLU,
Sigmoid, Tanh og LeakyReLU** og vis de fire test-accuracies i ét søjlediagram. Skabelonen
har allerede løkken — I skal udfylde trænings-rytmen og accuracy-målingen (mønstret fra
notebook 4).

In [ ]:
results = {}

for name, akt in [("ReLU", nn.ReLU()), ("Sigmoid", nn.Sigmoid()),
                  ("Tanh", nn.Tanh()), ("LeakyReLU", nn.LeakyReLU())]:
    torch.manual_seed(42)
    model = nn.Sequential(nn.Linear(784, 128), akt, nn.Linear(128, 10))
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    for epoch in range(3):
        for i in range(0, len(X_mnist), 64):
            optimizer.zero_grad()
            loss = loss_fn(model(X_mnist[i:i + 64]), y_mnist[i:i + 64])
            loss.backward()
            optimizer.step()

    with torch.no_grad():
        acc = (model(X_mnist_test).argmax(dim=1) == y_mnist_test).float().mean().item()
    results[name] = acc
    print(f"{name}: {acc:.1%}")

plt.bar(results.keys(), results.values())
plt.ylim(0.8, 1.0)
plt.ylabel("test-accuracy")
plt.title("Aktiveringsdysten (3 epoker)")
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Typisk resultat (3 epoker): ReLU/LeakyReLU/Tanh ligger tæt (~92 %), sigmoid halvandet procentpoint under. Bonus-observation: <code>nn.Sequential</code> er en genvej til små netværk — lagene køres bare i rækkefølge, uden at man selv skriver en klasse. Fint til eksperimenter; klassen giver mere kontrol. (Og bemærk seed-tricket, så alle fire starter med samme vægte — fair dyst!)</span>

##### Opgave 1.5 — Hyperparameter-jagten (ekstra)

Læringsrate og netværksstørrelse kaldes **hyperparametre** — tal, VI vælger, og som
modellen ikke selv lærer. I praksis prøver man sig systematisk frem. Nu skal pandas møde
PyTorch: kør en **dobbelt for-løkke** over læringsrater × skjulte størrelser (på et udsnit
af MNIST, så det går hurtigt), gem alle resultater i en **DataFrame**, og vis dem som et
**heatmap**. Udfyld hullerne — og find den bedste kombination!

In [ ]:
X_lille = X_mnist[:3000]
y_lille = y_mnist[:3000]

results = []
for lr in [0.0001, 0.001, 0.01]:
    for hidden in [16, 64, 256]:
        torch.manual_seed(42)
        model = nn.Sequential(nn.Linear(784, hidden), nn.ReLU(), nn.Linear(hidden, 10))
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        loss_fn = nn.CrossEntropyLoss()
        for epoch in range(2):
            for i in range(0, len(X_lille), 64):
                optimizer.zero_grad()
                loss = loss_fn(model(X_lille[i:i + 64]), y_lille[i:i + 64])
                loss.backward()
                optimizer.step()

        with torch.no_grad():
            acc = (model(X_mnist_test).argmax(dim=1) == y_mnist_test).float().mean().item()
        results.append({"lr": lr, "skjulte": hidden, "accuracy": acc})
        print(f"lr={lr}, skjulte={hidden}: {acc:.1%}")

res_df = pd.DataFrame(results)
table = res_df.pivot(index="lr", columns="skjulte", values="accuracy")
print(table)

plt.imshow(table, cmap="viridis")
plt.colorbar(label="accuracy")
plt.xticks(range(len(table.columns)), table.columns)
plt.yticks(range(len(table.index)), table.index)
plt.xlabel("skjulte neuroner")
plt.ylabel("læringsrate")
plt.title("Hyperparameter-jagten")
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Typisk vinder lr = 0.01 med 64 skjulte (~91 % på kun 2 epoker og 3.000 billeder), skarpt forfulgt af lr = 0.001 med 256; lr = 0.0001 når slet ikke i mål på så kort træning (helt ned til ~23 % med 16 skjulte!). Det her — en resultat-tabel + et heatmap — er PRÆCIS sådan, rigtige ML-folk vælger hyperparametre (bare med flere kombinationer og mere regnekraft). pandas og PyTorch er et par, ikke konkurrenter.</span>

##### Opgave 1.6 — Tegn selv et tal

Kan jeres netværk læse JERES håndskrift? Et 28×28-billede er bare et numpy-array, så I kan
"tegne" med slicing: `billede[raekker, kolonner] = 1.0` maler et hvidt felt. Startkoden
tegner et (meget kantet) 7-tal og spørger modellen.

**Jeres mission:** tegn mindst to andre cifre med slicing, og se om modellen kan læse dem.
Kig på sandsynlighederne — hvornår er den i tvivl? (Og hvis den fejler: er det jeres
håndskrift eller modellens skyld? )

In [ ]:
torch.manual_seed(42)
model_e6 = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))
optimizer = torch.optim.Adam(model_e6.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()
for epoch in range(5):
    for i in range(0, len(X_mnist), 64):
        optimizer.zero_grad()
        loss = loss_fn(model_e6(X_mnist[i:i + 64]), y_mnist[i:i + 64])
        loss.backward()
        optimizer.step()
print("model klar!")

# eksempel: et 1-tal, et 0 og et 4-tal tegnet med slicing
digits = {}

et_number = np.zeros((28, 28), dtype=np.float32)
et_number[4:24, 13:16] = 1.0                 # lodret streg
digits["1?"] = et_number

nul = np.zeros((28, 28), dtype=np.float32)
nul[4:7, 9:19] = 1.0                      # top
nul[21:24, 9:19] = 1.0                    # bund
nul[4:24, 7:10] = 1.0                     # venstre
nul[4:24, 18:21] = 1.0                    # højre
digits["0?"] = nul

fire = np.zeros((28, 28), dtype=np.float32)
fire[4:16, 8:11] = 1.0                    # venstre ben
fire[13:16, 8:21] = 1.0                   # tværstreg
fire[4:24, 17:20] = 1.0                   # højre ben
digits["4?"] = fire

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for axis, (name, image) in zip(axes, digits.items()):
    with torch.no_grad():
        point = model_e6(torch.tensor(image.reshape(1, 784)))
    pred = point.argmax().item()
    tillid = torch.softmax(point, dim=1).max().item()
    axis.imshow(image, cmap="gray")
    axis.set_title(f"tegnet: {name}  gæt: {pred} ({tillid:.0%})")
    axis.axis("off")
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Modellen rammer tit — men ikke altid! Kantede slicing-cifre ligner ikke helt MNIST's blødt håndskrevne (og MNIST-cifre er centreret og har gråtone-kanter), så modellen kan snuble på selv "pæne" tegninger. Det er en vigtig lektion om <b>distribution shift</b>: modeller er bedst på data, der LIGNER træningsdataene. Vil man have det vildere, kan man i Colab uploade et foto af et håndskrevet tal og skalere det ned til 28×28 — spørg en underviser.</span>

##### Opgave 1.7 — Fejl-detektiven (boss-level)

Pipeline-koden nedenfor skal træne en legendarisk-spotter (som notebook 2) — men der
gemmer sig **FEM fejl** fra hele emnet i den. Nogle crasher med en fejlbesked, andre
snyder i **total stilhed**. Find og ret alle fem!

*Tip: I har mødt hver eneste af fejlene før. Tjek: dataforberedelsen, modellens output,
loop-rytmen (×2) — og til sidst: hvad der egentlig bliver målt.*

In [ ]:
seks_stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
X_np = df[seks_stats].values.astype("float32")
X_np = (X_np - X_np.mean(axis=0)) / X_np.std(axis=0)     # FEJL 1: standardisering manglede (10.9)
y_np = df["Legendary"].astype(int).values.astype("float32")

X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(X_np, y_np, test_size=0.2, random_state=42)
X_train, X_test = torch.tensor(X_train_np), torch.tensor(X_test_np)
y_train, y_test = torch.tensor(y_train_np), torch.tensor(y_test_np)

class Spotter(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(6, 16)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(16, 1)
        self.sigmoid = nn.Sigmoid()                       # FEJL 2: BCELoss kræver sandsynligheder

    def forward(self, x):
        return self.sigmoid(self.layer2(self.activation(self.layer1(x))))

model = Spotter()
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(500):
    optimizer.zero_grad()                                 # FEJL 3: nulstilling manglede (7.5)
    y_hat = model(X_train).squeeze()
    loss = loss_fn(y_hat, y_train)
    loss.backward()                                        # FEJL 4: backward SKAL komme før step (10.3)
    optimizer.step()

with torch.no_grad():
    pred = (model(X_test).squeeze() > 0.5).float()        # FEJL 5: der blev målt på TRÆNINGSdata (10.11)
accuracy = (pred == y_test).float().mean()
print(f"accuracy: {accuracy.item():.1%}")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> De fem fejl: (1) ingen standardisering af X — stille fejl, træningen bliver dårlig; (2) ingen sigmoid på outputtet — BCELoss crasher med "all elements of input should be between 0 and 1"; (3) manglende zero_grad — gradienterne hober sig op; (4) step før backward — skridt uden friske gradienter; (5) accuracy målt på træningsdata — måler udenadslære, ikke kunnen. Rettet lander pipelinen på ~93-96 % test-accuracy. Hvis I fandt alle fem: tillykke, I fejlfinder nu på niveau med folk, der har lavet det her i årevis — de laver PRÆCIS de samme fejl.</span>

##### Opgave 1.8 — Frit valg fra Kaggle (ekstra)

Den sidste boss er friheden selv: **find jeres eget datasæt på
[kaggle.com/datasets](https://www.kaggle.com/datasets)** og træn et netværk på det.

**Tjekliste (= alt hvad I har lært):**
1. Find et datasæt med en CSV-fil og mindst én kolonne, der er værd at forudsige.
 Gode begynder-emner: vin-kvalitet, bilpriser, fisk-vægt, film-ratings...
2. Hent det: `kagglehub.dataset_download("ejer/datasæt-navn")` (navnet står i datasættets URL).
3. Udforsk og rens: `head`, `describe`, `isna().sum()` — drop eller udfyld huller,
 og vælg talkolonner som features.
4. Standardisér features. Train/test-split.
5. Regression eller klassifikation? → vælg output-lag og tabsfunktion (opgave 8.8!).
6. Byg model, træn med rytmen, plot tabskurven.
7. Mål på testsættet — og sammenlign ALTID med en doven baseline (gennemsnittet /
 den største klasse).

Skabelonen nedenfor er jeres startgerüst — udfyld den trin for trin.

In [ ]:
# ET eksempel: forudsig vinkvalitet (regression) fra kemiske målinger
path_own = kagglehub.dataset_download("uciml/red-wine-quality-cortez-et-al-2009")
import os
print(os.listdir(path_own))

own_df = pd.read_csv(path_own + "/winequality-red.csv")
print(own_df.shape, "| huller:", own_df.isna().sum().sum())

features = [k for k in own_df.columns if k != "quality"]
X_np = own_df[features].values.astype("float32")
X_np = (X_np - X_np.mean(axis=0)) / X_np.std(axis=0)
y_np = own_df["quality"].values.astype("float32")

X_tr, X_te, y_tr, y_te = train_test_split(X_np, y_np, test_size=0.2, random_state=42)
X_tr, X_te = torch.tensor(X_tr), torch.tensor(X_te)
y_tr, y_te = torch.tensor(y_tr), torch.tensor(y_te)

model = nn.Sequential(nn.Linear(len(features), 32), nn.ReLU(), nn.Linear(32, 1))
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
for epoch in range(2000):
    optimizer.zero_grad()
    loss = loss_fn(model(X_tr).squeeze(), y_tr)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    error = (model(X_te).squeeze() - y_te).abs().mean()
    lazy = (y_tr.mean() - y_te).abs().mean()      # baseline: gæt altid gennemsnittet
print(f"netværkets gennemsnitsfejl: {error.item():.2f} kvalitetspoint")
print(f"dovne baseline:             {lazy.item():.2f} kvalitetspoint")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Eksemplet (rødvin) lander typisk på ~0,5 kvalitetspoints fejl mod baselinens ~0,7 — netværket har lært NOGET, men vinkvalitet er svært (mennesker er også uenige!). Til underviseren: alt tæller her, bare tjeklisten følges — de klassiske begynderproblemer er tekstkolonner som features (skal droppes eller oversættes til tal) og glemte huller. Det er meningen, at de skal støde ind i dem og løse dem.</span>

In [ ]:
# Henter hjerte-data fra GitHub (Plan B: upload heart.csv via mappeikonet i Colab)
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/28-Data/MLData/heart.csv

import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
np.random.seed(42)

df = pd.read_csv("heart.csv")
df.head()

Hver række er en patient: alder, køn, blodtryk, kolesterol, maks-puls, brystsmerte-type
m.m. — og `HeartDisease` (1 = hjertesygdom). Tekstkolonnerne one-hot-koder vi med
`pd.get_dummies` (hver kategori får sin egen 0/1-kolonne), og så standardiserer vi,
som vi plejer:

In [ ]:
X = pd.get_dummies(df.drop(columns=["HeartDisease"])).astype("float32")
print("features efter one-hot:", X.shape)

X_np = X.values
X_np = (X_np - X_np.mean(axis=0)) / X_np.std(axis=0)     # standardisering, som altid
y_np = df["HeartDisease"].values.astype("float32")
print("andel med hjertesygdom:", round(float(y_np.mean()), 3))

# 2: Valideringssættet — den manglende brik

I Intro-ML delte vi data i **train** og **test**. Men der er et hul i den plan: når I
skruer på epoker, læringsrate og netstørrelse og måler på testsættet hver gang — så
vælger I til sidst præcis de indstillinger, der tilfældigvis passer TESTSÆTTET. I har
"slidt eksamen op": modellen består en prøve, den reelt har fået lov at øve sig på.

Derfor deler professionelle ALTID i tre:

| Sæt | Rolle | Skole-analogi |
|---|---|---|
| **Træning** (60 %) | modellen lærer af det | lektier |
| **Validering** (20 %) | VI vælger indstillinger efter det | terminsprøver |
| **Test** (20 %) | røres ÉN gang, til allersidst | eksamen |

Splittet laver vi med `train_test_split` to gange — først 60/40, så deles de 40 i to:

In [ ]:
X_train_np, X_rest, y_train_np, y_rest = train_test_split(X_np, y_np, test_size=0.4, random_state=42)
X_val_np, X_test_np, y_val_np, y_test_np = train_test_split(X_rest, y_rest, test_size=0.5, random_state=42)

X_train, X_val, X_test = map(torch.tensor, (X_train_np, X_val_np, X_test_np))
y_train, y_val, y_test = map(torch.tensor, (y_train_np, y_val_np, y_test_np))

print("træning:", len(X_train), "| validering:", len(X_val), "| test:", len(X_test))

## Fremprovokér overfitting!

Nu gør vi bevidst alt det, man ikke må: et KÆMPE netværk (over 70.000 parametre) på
BITTESMÅ data (550 patienter), trænet længe. Undervejs noterer vi tabet på både
træningsdata og valideringsdata — valideringstabet beregnes i `torch.no_grad()`, for
modellen må gerne blive MÅLT på valideringssættet, men aldrig LÆRE af det:

In [ ]:
def build_net():
    return nn.Sequential(
        nn.Linear(X.shape[1], 256), nn.ReLU(),
        nn.Linear(256, 256), nn.ReLU(),
        nn.Linear(256, 1), nn.Sigmoid())

model = build_net()
print("parametre:", sum(p.numel() for p in model.parameters()))

loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
train_loss, val_loss = [], []

for epoch in range(600):
    optimizer.zero_grad()
    loss = loss_fn(model(X_train).squeeze(), y_train)
    loss.backward()
    optimizer.step()

    with torch.no_grad():                          # mål, men lær ikke!
        train_loss.append(loss.item())
        val_loss.append(loss_fn(model(X_val).squeeze(), y_val).item())

In [ ]:
plt.plot(train_loss, label="træningstab")
plt.plot(val_loss, label="valideringstab")
plt.xlabel("epoke")
plt.ylabel("BCE-tab")
plt.legend()
plt.title("Den vigtigste figur i praktisk deep learning")
plt.show()

print(f"laveste valideringstab: {min(val_loss):.3f} ved epoke {val_loss.index(min(val_loss))}")

**DÉR er overfitting.** Træningstabet falder og falder (modellen lærer patienterne
udenad) — men valideringstabet bunder omkring epoke ~130 og **stiger derefter**. Fra
det punkt bliver modellen bedre til at HUSKE og dårligere til at FORUDSIGE. Al træning
efter bunden er ikke bare spildt — den er skadelig.

## Early stopping: stop, mens legen er god

Det oplagte modtræk: gem modellen, hver gang valideringstabet sætter ny bundrekord —
og brug til sidst den gemte udgave i stedet for den sidste:

In [ ]:
torch.manual_seed(42)
model = build_net()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

best_val_loss = 999.0
best_epoch = 0
best_weights = None

for epoch in range(600):
    optimizer.zero_grad()
    loss = loss_fn(model(X_train).squeeze(), y_train)
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        val_now = loss_fn(model(X_val).squeeze(), y_val).item()
    if val_now < best_val_loss:                              # ny bundrekord?
        best_val_loss = val_now
        best_epoch = epoch
        best_weights = copy.deepcopy(model.state_dict())    # gem en KOPI af vægtene

model.load_state_dict(best_weights)                         # spol tilbage til rekorden
print(f"bedste model: epoke {best_epoch} (valideringstab {best_val_loss:.3f})")

with torch.no_grad():
    val_acc = ((model(X_val).squeeze() > 0.5).float() == y_val).float().mean()
print(f"validerings-accuracy: {val_acc.item():.1%}")

(`model.state_dict()` er en ordbog med alle modellens vægte, og `copy.deepcopy` tager
en RIGTIG kopi — ellers ville vores "gemte" vægte bare pege på de levende vægte, der
ændrer sig videre. `load_state_dict` lægger dem tilbage.)

### Opgaver

##### Opgave 2.1
Aflæs U-kurven fra det store eksperiment: cirka ved hvilken epoke begynder
valideringstabet at stige? Og hvad er der galt med at sige *"men træningstabet falder
jo stadig — modellen bliver da bedre!"*?

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Valideringstabet bunder omkring epoke 120-140 og kravler derefter opad. Det faldende træningstab efter det punkt måler kun, hvor godt modellen HUSKER de 550 patienter, den har set — ikke hvor god den er til nye. Fremskridt på træningsdata er kun ægte fremskridt, så længe valideringskurven følger med ned.</span>

##### Opgave 2.2
Hvorfor kan vi ikke bare bruge TESTSÆTTET til at vælge antal epoker (og droppe
valideringssættet)? Hvad ville der ske med troværdigheden af vores allersidste
test-måling?

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Så har vi valgt netop de indstillinger, der maksimerer accuracy på PRÆCIS de 184 test-patienter — testsættet er ikke længere "nye, aldrig sete data", og målingen bliver systematisk for optimistisk. Hver gang man kigger på testsættet og justerer noget bagefter, lækker der lidt facit. Terminsprøver (validering) er til at justere efter — eksamen (test) tages én gang.</span>

##### Opgave 2.3
Udfyld de to `test_size`-værdier, så splittet bliver præcis 60/20/20. (Tænk: andet
split deler RESTEN — hvor stor en andel af de 40 % skal blive til test?)

In [ ]:
X_a, X_r, y_a, y_r = train_test_split(X_np, y_np, test_size=0.4, random_state=42)
X_v, X_t, y_v, y_t = train_test_split(X_r, y_r, test_size=0.5, random_state=42)
print(len(X_a), len(X_v), len(X_t))

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Første split: 40 % lægges til side. Andet split: HALVDELEN af resten (test_size=0.5) bliver test — altså 20 % af det hele. Klassisk lommeregner-fælde: at skrive 0.2 i det andet split (det giver kun 8 % test).</span>

##### Opgave 2.4
Gentag overfitting-eksperimentet med et LILLE net: skift alle 256-taller ud med 16.
Hvordan ser kurverne ud nu — og hvorfor giver det mening, at et lille net overfitter
mindre?

In [ ]:
torch.manual_seed(42)
model_lille = nn.Sequential(
    nn.Linear(X.shape[1], 16), nn.ReLU(),
    nn.Linear(16, 16), nn.ReLU(),
    nn.Linear(16, 1), nn.Sigmoid())
optimizer = torch.optim.Adam(model_lille.parameters(), lr=0.0001)
train_loss2, val_loss2 = [], []
for epoch in range(600):
    optimizer.zero_grad()
    loss = loss_fn(model_lille(X_train).squeeze(), y_train)
    loss.backward()
    optimizer.step()
    with torch.no_grad():
        train_loss2.append(loss.item())
        val_loss2.append(loss_fn(model_lille(X_val).squeeze(), y_val).item())

plt.plot(train_loss2, label="træningstab")
plt.plot(val_loss2, label="valideringstab")
plt.legend()
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Med 16 neuroner følges de to kurver pænt ad — nettet har simpelthen ikke kapacitet til at lære 550 patienter udenad. Modellens STØRRELSE er i sig selv et overfitting-værktøj: det mindste net, der kan løse opgaven, er tit det bedste valg på små data.</span>

##### Opgave 2.5 (find fejlen)
Denne kammerats "valideringskurve" ser mistænkeligt godt ud — den falder bare og
falder, intet U! Kig godt på linjen, hvor valideringstabet beregnes. Hvad er der sket —
og hvorfor er netop DEN fejl så farlig? (Den giver jo ingen fejlbesked...)

In [ ]:
torch.manual_seed(42)
model_k = build_net()
optimizer = torch.optim.Adam(model_k.parameters(), lr=0.0001)
val_loss_k = []
for epoch in range(600):
    optimizer.zero_grad()
    loss = loss_fn(model_k(X_train).squeeze(), y_train)
    loss.backward()
    optimizer.step()
    with torch.no_grad():
        val_loss_k.append(loss_fn(model_k(X_val).squeeze(), y_val).item())   # ← X_val og y_val!

plt.plot(val_loss_k, label="valideringstab")
plt.legend()
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Copy-paste-klassikeren: "valideringstabet" blev beregnet på <code>X_traen/y_traen</code> — altså bare træningstabet én gang til. Kurven ser perfekt ud, koden kører fejlfrit, og man tror, man har verdens bedste model. Stille fejl som denne er farligere end crashes — tjek ALTID, at valideringskurven kommer fra valideringsdata (lakmusprøve: den skal ligge OVER træningskurven det meste af tiden).</span>

##### Opgave 2.6
Overfitting elsker små datasæt. Brug kun de første 275 træningspatienter
(`X_traen[:275]`, `y_traen[:275]`) og kør det store net igen. Kommer U-kurvens bund
tidligere eller senere — og bliver gabet mellem kurverne større eller mindre?

In [ ]:
torch.manual_seed(42)
model_h = build_net()
optimizer = torch.optim.Adam(model_h.parameters(), lr=0.0001)
train_loss3, val_loss3 = [], []
for epoch in range(600):
    optimizer.zero_grad()
    loss = loss_fn(model_h(X_train[:275]).squeeze(), y_train[:275])
    loss.backward()
    optimizer.step()
    with torch.no_grad():
        train_loss3.append(loss.item())
        val_loss3.append(loss_fn(model_h(X_val).squeeze(), y_val).item())

plt.plot(train_loss3, label="træningstab (275 patienter)")
plt.plot(val_loss3, label="valideringstab")
plt.legend()
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Med halvt så mange patienter bunder valideringstabet tidligere og stiger stejlere, og gabet mellem kurverne vokser — 275 eksempler er endnu lettere at lære udenad end 550. Tommelfingerregel: jo mindre data, desto mere regularisering (og mindre net). Mere data er i øvrigt den bedste "regularisering", der findes.</span>

##### Opgave 2.7
Udfyld de tre manglende linjer i early stopping (gem rekorden, epoken og en KOPI af
vægtene).

In [ ]:
torch.manual_seed(42)
model_es = build_net()
optimizer = torch.optim.Adam(model_es.parameters(), lr=0.0001)
best_val_loss = 999.0
best_epoch = 0
best_weights = None

for epoch in range(600):
    optimizer.zero_grad()
    loss = loss_fn(model_es(X_train).squeeze(), y_train)
    loss.backward()
    optimizer.step()
    with torch.no_grad():
        val_now = loss_fn(model_es(X_val).squeeze(), y_val).item()
    if val_now < best_val_loss:
        best_val_loss = val_now
        best_epoch = epoch
        best_weights = copy.deepcopy(model_es.state_dict())

model_es.load_state_dict(best_weights)
print(f"bedste epoke: {best_epoch} (val-tab {best_val_loss:.3f})")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Uden <code>copy.deepcopy</code> gemmer man kun en HENVISNING til de levende vægte — og så "opdaterer" ens gemte rekord sig selv, hver gang modellen træner videre. En af de mest udbredte stille fejl i hjemmelavede træningsloops.</span>

##### Opgave 2.8 (ekstra)
Rigtig early stopping venter ikke til epoke 600 — den AFBRYDER, når valideringstabet
ikke har sat bundrekord i `taalmodighed` epoker i træk. Udfyld tæller-logikken
(nulstil ved rekord, tæl op ellers). Hvor mange epoker sparede I?

In [ ]:
torch.manual_seed(42)
model_p = build_net()
optimizer = torch.optim.Adam(model_p.parameters(), lr=0.0001)
best_val_loss = 999.0
patience = 50
siden_rekord = 0

for epoch in range(600):
    optimizer.zero_grad()
    loss = loss_fn(model_p(X_train).squeeze(), y_train)
    loss.backward()
    optimizer.step()
    with torch.no_grad():
        val_now = loss_fn(model_p(X_val).squeeze(), y_val).item()
    if val_now < best_val_loss:
        best_val_loss = val_now
        siden_rekord = 0
    else:
        siden_rekord = siden_rekord + 1
    if siden_rekord >= patience:
        print(f"early stopping ved epoke {epoch}!")
        break

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Med tålmodighed 50 stopper løbet typisk omkring epoke 180-250 — altså 350+ sparede epoker. Tålmodigheden findes, fordi valideringskurven er hakkende: ét dårligt punkt betyder ikke, at festen er slut. (<code>break</code> hopper ud af for-løkken — sådan stopper man tidligt.)</span>

# 3: Værktøjskassen — dropout, weight decay & batch norm

Early stopping behandler symptomet (stop før udenadslæren). De næste værktøjer angriber
årsagen: de gør det SVÆRERE for nettet at lære udenad.

For ikke at skrive det samme loop fem gange pakker vi det i en funktion — læs den, det
er jeres velkendte rytme plus én ny detalje: **`model.train()` og `model.eval()`**.
De to kald tænder og slukker for "træningsopførsel". Hvorfor det pludselig er vigtigt,
ser I om lidt:

In [ ]:
def train_with_curves(model, epochs=600, lr=0.0001, weight_decay=0.0):
    loss_fn = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    train_loss, val_loss = [], []
    for epoch in range(epochs):
        model.train()                                # træningsopførsel TIL
        optimizer.zero_grad()
        loss = loss_fn(model(X_train).squeeze(), y_train)
        loss.backward()
        optimizer.step()

        model.eval()                                 # træningsopførsel FRA, før vi måler
        with torch.no_grad():
            train_loss.append(loss_fn(model(X_train).squeeze(), y_train).item())
            val_loss.append(loss_fn(model(X_val).squeeze(), y_val).item())
    return train_loss, val_loss


def val_accuracy(model):
    model.eval()
    with torch.no_grad():
        return ((model(X_val).squeeze() > 0.5).float() == y_val).float().mean().item()

## Dropout: træn med huller i holdet

**Dropout** slukker i hvert træningsskridt en tilfældig andel af neuronerne (fx 50 %).
Så kan ingen enkelt neuron få lov at huske hele facitlisten — holdet tvinges til at
sprede viden ud. Som fodboldtræning, hvor tilfældige holdkammerater udebliver hver
gang: alle ender med at kunne alle positioner.

Vigtigt: hullerne er KUN til træning. Når modellen skal bruges, spiller hele holdet —
det er derfor `model.eval()` findes. I PyTorch er dropout bare et lag:

In [ ]:
torch.manual_seed(42)
model_dropout = nn.Sequential(
    nn.Linear(X.shape[1], 256), nn.ReLU(), nn.Dropout(0.5),
    nn.Linear(256, 256), nn.ReLU(), nn.Dropout(0.5),
    nn.Linear(256, 1), nn.Sigmoid())

train_loss_d, val_loss_d = train_with_curves(model_dropout)

plt.plot(val_loss, label="uden dropout (fra afsnit 3)")
plt.plot(val_loss_d, label="med dropout 0.5")
plt.xlabel("epoke")
plt.ylabel("valideringstab")
plt.legend()
plt.show()
print(f"validerings-accuracy med dropout: {val_accuracy(model_dropout):.1%}")

## Weight decay: hold vægtene små

Udenadslære kræver typisk store, ekstreme vægte ("HVIS præcis denne kombination, SÅ
syg!"). **Weight decay** lægger en lille straf på store vægte oveni tabet, så nettet
foretrækker små vægte — og dermed enkle, glatte løsninger. I PyTorch er det
bogstaveligt talt ét ekstra argument til optimizeren:

In [ ]:
torch.manual_seed(42)
model_wd = build_net()
train_loss_w, val_loss_w = train_with_curves(model_wd, weight_decay=0.01)

plt.plot(val_loss, label="uden noget")
plt.plot(val_loss_w, label="med weight decay 0.01")
plt.xlabel("epoke")
plt.ylabel("valideringstab")
plt.legend()
plt.show()
print(f"validerings-accuracy med weight decay: {val_accuracy(model_wd):.1%}")

## Batch normalization: stabilisatoren

**Batch norm** standardiserer tallene MELLEM lagene (jeres standardiserings-trick,
bare gentaget inde i nettet). Det gør træning af DYBE netværk markant mere stabil og
hurtig, og næsten alle store moderne net bruger det.

Men lad os være ærlige og AFPRØVE det i stedet for at tro på reklamen:

In [ ]:
torch.manual_seed(42)
model_bn = nn.Sequential(
    nn.Linear(X.shape[1], 256), nn.BatchNorm1d(256), nn.ReLU(),
    nn.Linear(256, 256), nn.BatchNorm1d(256), nn.ReLU(),
    nn.Linear(256, 1), nn.Sigmoid())

train_loss_b, val_loss_b = train_with_curves(model_bn)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_loss, label="uden")
axes[0].plot(train_loss_b, label="med batch norm")
axes[0].set_title("træningstab")
axes[0].legend()
axes[1].plot(val_loss, label="uden")
axes[1].plot(val_loss_b, label="med batch norm")
axes[1].set_title("valideringstab")
axes[1].legend()
plt.show()

Se godt efter: med batch norm **styrtdykker træningstabet** (stabilisatoren virker!)
— men valideringstabet vender TIDLIGERE og stiger STEJLERE. På et lillebitte datasæt
som vores accelererer batch norm bare udenadslæren. Lærdommen er vigtig: batch norm er
et værktøj til at få STORE, DYBE net til at træne stabilt — det er ikke et middel mod
overfitting. Kend jeres værktøjs job.

## Den samlede opskrift

I praksis kombinerer man: et fornuftigt net + dropout + lidt weight decay + early
stopping. Lad os køre hele pakken — og til allersidst, én gang, røre testsættet:

In [ ]:
torch.manual_seed(42)
model_fuld = nn.Sequential(
    nn.Linear(X.shape[1], 256), nn.ReLU(), nn.Dropout(0.5),
    nn.Linear(256, 256), nn.ReLU(), nn.Dropout(0.5),
    nn.Linear(256, 1), nn.Sigmoid())

loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model_fuld.parameters(), lr=0.0001, weight_decay=0.01)
best_val_loss = 999.0
best_weights = None

for epoch in range(600):
    model_fuld.train()
    optimizer.zero_grad()
    loss = loss_fn(model_fuld(X_train).squeeze(), y_train)
    loss.backward()
    optimizer.step()

    model_fuld.eval()
    with torch.no_grad():
        val_now = loss_fn(model_fuld(X_val).squeeze(), y_val).item()
    if val_now < best_val_loss:
        best_val_loss = val_now
        best_weights = copy.deepcopy(model_fuld.state_dict())

model_fuld.load_state_dict(best_weights)
model_fuld.eval()
with torch.no_grad():
    test_acc = ((model_fuld(X_test).squeeze() > 0.5).float() == y_test).float().mean()
print(f"ENDELIG test-accuracy (målt én gang!): {test_acc.item():.1%}")

| Værktøj | Hvad gør det? | Hvornår? |
|---|---|---|
| Valideringssæt | ærlig måling undervejs | ALTID |
| Early stopping | stop før udenadslæren | næsten altid |
| Dropout | sluk neuroner → tving samarbejde | mellemstore/store net |
| Weight decay | straf store vægte → enkle løsninger | næsten altid (små doser) |
| Batch norm | stabilisér træning af DYBE net | dybe net, store datasæt |
| Mindre net / mere data | angrib årsagen direkte | når muligt! |

### Opgaver

##### Opgave 3.1
Prøv dropout på **0.1, 0.5 og 0.9** og sammenlign valideringskurverne i ét plot. Hvad
går der galt ved 0.9 — og hvad hedder det modsatte af overfitting?

In [ ]:
for rate in [0.1, 0.5, 0.9]:
    torch.manual_seed(42)
    m = nn.Sequential(
        nn.Linear(X.shape[1], 256), nn.ReLU(), nn.Dropout(rate),
        nn.Linear(256, 256), nn.ReLU(), nn.Dropout(rate),
        nn.Linear(256, 1), nn.Sigmoid())
    t_loss, v_loss = train_with_curves(m)
    plt.plot(v_loss, label=f"dropout {rate}")
plt.xlabel("epoke")
plt.ylabel("valideringstab")
plt.legend()
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> 0.1: næsten som ingenting. 0.5: pæn, flad kurve — sweet spot. 0.9: nettet lærer aldrig rigtig noget — 90 % af holdet mangler til hver træning. Det hedder <b>underfitting</b>: modellen er (blevet gjort) for svag til opgaven. Regularisering er en balance mellem to grøfter.</span>

##### Opgave 3.2 (find fejlen)
En kammerat måler accuracy på dropout-modellen — og får et NYT tal, hver gang cellen
køres?! Og tallene er lavere end forventet. Kør cellen 3-4 gange og se selv. Én linje
er forkert — hvilken, og hvorfor ændrer den alt?

In [ ]:
model_dropout.eval()    # ← SLUK dropout, før der måles!
with torch.no_grad():
    acc = ((model_dropout(X_val).squeeze() > 0.5).float() == y_val).float().mean()
print(f"accuracy: {acc.item():.1%}   — nu er tallet det samme hver gang")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> I træningstilstand er dropout AKTIV — modellen slukker tilfældige neuroner, selv når man bare måler. Hvert kald giver derfor nye tilfældige huller og dermed nye (og dårligere) svar. <code>model.eval()</code> slukker dropout (og sætter batch norm i måletilstand). Reglen: <b>train() når der trænes, eval() når der måles.</b> Denne fejl er formentlig den hyppigste PyTorch-fejl overhovedet — nu er I vaccineret.</span>

##### Opgave 3.3
Udfyld `weight_decay`-argumentet (prøv 0.01), og tjek at valideringskurven bliver
fladere end uden.

In [ ]:
torch.manual_seed(42)
m = build_net()
t_loss, v_loss = train_with_curves(m, weight_decay=0.01)
plt.plot(val_loss, label="uden")
plt.plot(v_loss, label="med weight decay")
plt.legend()
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> <code>torch.optim.Adam(..., weight_decay=0.01)</code>. Kurven bliver tydeligt fladere efter bunden. (Lærernote: teknisk set implementerer AdamW weight decay mere korrekt end Adam — på dette niveau er forskellen ligegyldig, men nævn den gerne for nysgerrige.)</span>

##### Opgave 3.4
Weight decay har også en overdosis: prøv **0, 0.0001, 0.01 og 1.0** og sammenlign
kurver/accuracy. Hvad sker der ved 1.0, og hvorfor?

In [ ]:
for wd in [0.0, 0.0001, 0.01, 1.0]:
    torch.manual_seed(42)
    m = build_net()
    t_loss, v_loss = train_with_curves(m, weight_decay=wd)
    print(f"wd = {wd}: validerings-accuracy {val_accuracy(m):.1%}")
    plt.plot(v_loss, label=f"wd = {wd}")
plt.legend()
plt.ylabel("valideringstab")
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> 0.0001 gør næsten ingenting, 0.01 er sweet spot, og ved 1.0 bliver straffen for store vægte så voldsom, at alle vægte presses mod nul — modellen kan ikke længere repræsentere noget som helst (underfitting igen, nu ad kemisk vej). Samme mønster som dropout 0.9: hvert værktøj har en dosis.</span>

##### Opgave 3.5
Modellen skal bruges på rigtige patienter. Diskutér: hvad koster en **falsk negativ**
(syg patient, modellen siger rask) i forhold til en **falsk positiv** (rask patient,
modellen siger syg)? Og hvad kunne man gøre ved 0,5-grænsen, hvis man hellere vil have
den ene slags fejl end den anden?

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

<span style="color:red"><b>LØSNINGSFORSLAG:</b> En falsk negativ kan koste et liv (patienten sendes hjem uden behandling); en falsk positiv koster en ekstra undersøgelse og en forskrækkelse. De to fejl er altså vildt asymmetriske — og derfor kan man SÆNKE grænsen (fx til 0,3): modellen råber op ved mindre mistanke, fanger flere syge og accepterer flere falske alarmer. Grænseværdien er en ETISK beslutning, ikke en teknisk — og den bør ikke træffes af den, der har skrevet koden, alene.</span>

##### Opgave 3.6
Byg jeres bedste opskrift: kombinér valgfrit netværksstørrelse, dropout, weight decay
og early stopping — men mål KUN på valideringssættet, mens I eksperimenterer. Når I har
valgt jeres endelige opskrift: mål på testsættet ÉN gang. Hvor tæt ligger jeres
validerings- og test-accuracy?

In [ ]:
# eksperimentér her (mål kun på validering!) — og til allersidst: én test-måling
torch.manual_seed(42)
m = nn.Sequential(
    nn.Linear(X.shape[1], 256), nn.ReLU(), nn.Dropout(0.5),
    nn.Linear(256, 256), nn.ReLU(), nn.Dropout(0.5),
    nn.Linear(256, 1), nn.Sigmoid())
t_loss, v_loss = train_with_curves(m, weight_decay=0.01)
print(f"validering: {val_accuracy(m):.1%}")

# NÅR (og kun når) I er færdige med at eksperimentere:
# m.eval()
# with torch.no_grad():
#     test_acc = ((m(X_test).squeeze() > 0.5).float() == y_test).float().mean()
# print(f"test: {test_acc.item():.1%}")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Alt fornuftigt godkendes. Typisk lander både validering og test omkring 84-88 %, og de ligger tæt på hinanden, HVIS man har holdt fingrene fra testsættet undervejs — det er hele pointen. Ligger test markant UNDER validering, har man (måske uden at ville det) tilpasset sig valideringssættet gennem mange eksperimenter — det har også et navn: validerings-overfitting. Turtles all the way down. </span>

##### Opgave 3.7 (ekstra)
Batch norm fik hårde ord ovenfor — men den lovede også HURTIGERE træning. Mål det:
hvor mange epoker tager det henholdsvis med og uden batch norm at få træningstabet
under 0.3? (Skriv en for-løkke, der finder den første epoke under grænsen.)

In [ ]:
torch.manual_seed(42)
m_uden = build_net()
t_uden, v_uden = train_with_curves(m_uden)

torch.manual_seed(42)
m_med = nn.Sequential(
    nn.Linear(X.shape[1], 256), nn.BatchNorm1d(256), nn.ReLU(),
    nn.Linear(256, 256), nn.BatchNorm1d(256), nn.ReLU(),
    nn.Linear(256, 1), nn.Sigmoid())
t_med, v_med = train_with_curves(m_med)

for name, curve in [("uden batch norm", t_uden), ("med batch norm", t_med)]:
    foerste = None
    for epoch, loss in enumerate(curve):
        if loss < 0.3:
            foerste = epoch
            break
    print(f"{name}: træningstab under 0.3 ved epoke {foerste}")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Batch norm når typisk under 0.3 flere gange hurtigere (ofte epoke ~30-60 mod ~150-250 uden). Så reklamen taler sandt om HASTIGHED — problemet på vores lillebitte datasæt er kun, at den hastighed bruges på at overfitte hurtigere. På ImageNet-størrelse data er hurtigere træning guld værd. Værktøj + kontekst = værdi.</span>

##### Opgave 3.8 (ekstra)
Tilbage til Intro-ML-tricket: gennemsnittet af en True/False-række. Beregn hvor mange af
testsættets FAKTISK SYGE patienter jeres endelige model fanger (recall) — og sammenlign
med den samlede accuracy. Hvilket af de to tal ville I vise til en læge først?

In [ ]:
model_fuld.eval()
with torch.no_grad():
    pred = (model_fuld(X_test).squeeze() > 0.5).float()

sick = y_test == 1
recall = pred[sick].mean().item()
print(f"accuracy: {(pred == y_test).float().mean().item():.1%}")
print(f"recall:   {recall:.1%}")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> <code>gaet[syge].mean()</code> — blandt de faktisk syge er gættene 0/1, så gennemsnittet er netop andelen, der fanges. Recall er tallet, en læge vil se først: "hvor mange syge overser systemet?" er vigtigere end den flatterende samlede accuracy (jf. den dovne Pokémon-model i Intro-ML!).</span>